# Chapter 14 &mdash; Semi-Deciders, and Why RE Languages Are Procedures

**Concept 6 of the Chapter 14 decomposition:** *Semi-Deciders, and Why RE Languages Are Procedures*

A semi-decider halts and says "yes" only when the answer is yes; otherwise you wait, possibly forever.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Semi-Deciders/Concept-Semi-Deciders.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A **semi-decider** for $L$:

* on $w \in L$ it **halts and accepts**;
* on $w \notin L$ it may reject, or may **run forever**.

That is exactly "$L$ is RE". The machine implements a **procedure**, not an algorithm.

The practical shape of a semi-decider is a **search**: enumerate candidate witnesses
and stop when one works. If a witness exists you will find it; if none does, you search
forever.

And the crucial usage note: **a semi-decider's silence is not a 'no'.** Any system
built on one must have a story for "still running" that is not "therefore false".

## 2. Definitions

### A search-shaped semi-decider

In [ ]:
def semi_decide_has_factor(n, limit=None):
    # semi-decides "n is composite" by searching for a factor
    k, steps = 2, 0
    while limit is None or steps < limit:
        steps += 1
        if k * k > n: 
            if limit is None: return (False, steps)     # only terminates for small n
            return (None, steps)
        if n % k == 0: return (True, steps)
        k += 1
    return (None, steps)                                 # 'not yet'

### A TM semi-decider, for contrast with a decider

In [ ]:
Semi = md2mc('''TM
!! accepts tapes containing a 1; loops forever on 0^n
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> I
I : . ; . , R -> I
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

<!-- nav-strip -->

---

&larr;&nbsp;[Ch14&nbsp;5.&nbsp;A Recursive Language: $L_{EmptyDFA}$, and the Definition of a Decider](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-L-EmptyDFA-Is-Recursive/Concept-L-EmptyDFA-Is-Recursive.ipynb) &nbsp;&middot;&nbsp; [**Chapter 14** index](https://github.com/ganeshutah/Jove/blob/master/Chapter14/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;7.&nbsp;Combining Semi-Deciders for $L$ and $\overline{L}$, and the "No Wimp" Clause](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Combining-Semi-Deciders/Concept-Combining-Semi-Deciders.ipynb)&nbsp;&rarr;

---

## 3. Tests

On a member, the semi-decider halts and says yes.

In [ ]:
for n in [9, 15, 91]:
    ans, steps = semi_decide_has_factor(n, limit=50)
    print("  %-4d composite? %-6s after %d steps" % (n, ans, steps))
assert semi_decide_has_factor(9, limit=50)[0]

On a non-member, you get 'not yet' &mdash; for as long as you care to wait.

In [ ]:
for limit in [3, 10, 40]:
    ans, steps = semi_decide_has_factor(101, limit=limit)
    print("  limit %2d : 101 composite? %-6s after %d steps" % (limit, ans, steps))
print("\nEvery answer is None.  None is not False.")

The TM version behaves the same way.

In [ ]:
print("'001' contains a 1 :", tm_halts(Semi, '001', fuel=100),
      " accepted :", tm_accepts(Semi, '001', fuel=100))
print("'000' contains no 1:", tm_halts(Semi, '000', fuel=500))
assert tm_accepts(Semi, '001', fuel=100)
assert not tm_halts(Semi, '000', fuel=500)

**Silence is not a 'no'.** This is the mistake to design against.

In [ ]:
print("observed: the machine has not answered")
print("possible: the answer is 'no'")
print("possible: the answer is 'yes' and it needs one more step")
print()
print("A system that treats 'no answer yet' as 'no' is unsound.")
print("A system that waits is unresponsive.  Pick your poison deliberately.")

Search is the canonical semi-decider shape.

In [ ]:
SHAPES = [("is n composite",        "search for a factor"),
          ("does M accept w",       "run M on w"),
          ("is this formula SAT",   "search for an assignment"),
          ("do G1 and G2 differ",   "enumerate strings in numeric order (Concept 8)")]
for q, how in SHAPES: print("  %-24s %s" % (q, how))
print("\nEach halts when it FINDS something.  None halts when there is nothing.")

## 4. Exercises


1. Turn the composite semi-decider into a decider. What did you have to know?
2. Which of the four searches above can be turned into deciders?
3. Why is "run $M$ on $w$" a semi-decider for $A_{TM}$ and not a decider?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter14/Concept-Semi-Deciders')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')